# Vocabulary Evaluator

**The Vocabulary Evaluator** helps determine whether texts use words that align with grade-level
expectations and support growth in academic language. It runs in two steps:

1. **Background knowledge** — the model generates a short assumption about what a student at the
   target grade already knows about the text's topic, from a shared prompt (`background-knowledge.txt`).
2. **Vocabulary complexity** — a grade-specific prompt + model rates the vocabulary, returning:
   * **complexity_score**: `slightly_complex` to `exceedingly_complex`
   * **tier_2_words** / **tier_3_words** / **archaic_words** / **other_complex_words**: word-level breakdown
   * **reasoning**: why the text fits the chosen level

## One config, two step variants

Grades 3-4 and grades 5-12 use genuinely different rubrics, models, and (for grades 3-4 only) a
Flesch-Kincaid preprocessing signal — not a stylistic split, two independently-validated prompt designs.
One `config.json` declares both `vocab_complexity_grades_3_4` and `vocab_complexity_other_grades` as
separate steps, each with a `condition` on `grade_level` saying which grades it applies to. This
notebook evaluates each step's condition against the actual input to pick the right one, the same
generic mechanism any consumer (including the published SDKs) can use.

> **Known limitation carried over from the SDKs, not introduced here:** `other-grades-user.txt`'s
> final line tells the model to *"use only a single integer... don't include any other text."*
> `output_schema.json` requires a full JSON object with a string `complexity_score` instead. This is a
> real, verified contradiction (see the `vocab_complexity_other_grades` step description in
> `config.json` for the captured evidence) — left as-is per maintainer direction. Structured output
> below enforces the true (JSON) contract regardless of what that line says.


In [ ]:
%pip install -qU langchain-google-genai langchain-openai langchain python-dotenv textstat

In [ ]:
import getpass
import hashlib
import json
import os
import pprint as pp
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
import textstat

In [ ]:
# Check for API keys. Background knowledge and the other-grades complexity step
# use OpenAI; the grades-3-4 complexity step uses Google.
load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")


### Load the config

`config.json` is the source of truth. Prompt files are loaded from disk and verified against the
`sha256` recorded for each step's messages.

> `input_schema.json` is a CI-side contract only (`scripts/check.py` binds each fixture's `input` to
> it) and isn't needed at runtime, so only `output_schema.json` is loaded here — it drives structured
> output for both `vocab_complexity_*` steps.


In [ ]:
def load_config():
    """Load config.json + hash-verified prompts for every step + output schema."""
    with open("config.json") as f:
        config = json.load(f)
    with open("output_schema.json") as f:
        output_schema = json.load(f)

    steps = {}
    for step in config["steps"]:
        messages = []
        for msg_spec in step["prompt"]["messages"]:
            path = Path(msg_spec["source_path"])
            text = path.read_text()
            actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
            declared_sha = msg_spec["sha256"]
            assert actual_sha == declared_sha, (
                f"prompt drift detected in step {step['id']!r} "
                f"role={msg_spec['role']!r} ({msg_spec['source_path']}): "
                f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
            )
            messages.append((msg_spec["role"], text))
        steps[step["id"]] = {"spec": step, "messages": messages}

    return config, output_schema, steps


CONFIG, OUTPUT_SCHEMA, STEPS = load_config()

print(f"Loaded {CONFIG['evaluator']['id']}")
for step_id, step in STEPS.items():
    model = step["spec"]["model"]
    condition = step["spec"].get("condition")
    cond_str = f"  condition={condition}" if condition else ""
    sha_list = ", ".join(
        f"{role}={hashlib.sha256(text.encode()).hexdigest()[:12]}"
        for role, text in step["messages"]
    )
    print(f"    {step_id:<28} model={model['provider']}/{model['name']:<16} {sha_list}{cond_str}")


### Step selection via condition

Every step or preprocessing entry may declare a `condition` (`{"input": ..., "in": [...]}`). A step
with no condition always applies (`background_knowledge`, shared by both grade bands); a step with a
condition only applies when the named input's value is one of the listed values. Picking the right
`vocab_complexity_*` step for a given `grade_level` is just evaluating that condition generically —
no hardcoded grade-range branching.


In [ ]:
def condition_matches(condition, inputs):
    """A step/preprocessing entry with no condition always applies."""
    if condition is None:
        return True
    value = str(inputs.get(condition["input"]))
    return value in [str(v) for v in condition["in"]]


def select_step(candidate_ids, inputs):
    """Pick whichever of these step ids has a matching condition for inputs."""
    matches = [
        step_id for step_id in candidate_ids
        if condition_matches(STEPS[step_id]["spec"].get("condition"), inputs)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"expected exactly one matching step among {candidate_ids} for {inputs}, "
            f"got {len(matches)}: {matches}"
        )
    return STEPS[matches[0]]


### Step 1: background knowledge (shared by both branches)

In [ ]:
def make_llm(model_spec, temperature):
    if model_spec["provider"] == "google":
        return ChatGoogleGenerativeAI(model=model_spec["name"], temperature=temperature)
    if model_spec["provider"] == "openai":
        return ChatOpenAI(model=model_spec["name"], temperature=temperature)
    raise ValueError(f"unsupported provider: {model_spec['provider']!r}")


def generate_background_knowledge(text: str, grade: str) -> str:
    step = STEPS["background_knowledge"]
    llm = make_llm(step["spec"]["model"], step["spec"]["generation"]["temperature"])
    # Single user-role message; background-knowledge.txt uses {text} and {grade_level}.
    # This file legitimately contains literal { } (an embedded JSON topics blob), so
    # substitution must be plain replace(), not str.format() -- matches the TS SDK's
    # getBackgroundKnowledgePrompt(), which uses .replaceAll() for the same reason.
    role, template = step["messages"][0]
    assert role == "user"
    prompt = template.replace("{text}", text).replace("{grade_level}", grade)
    return llm.invoke(prompt).content.strip()


### Step 2: vocabulary complexity (grade-specific)

Structured output is driven by each branch's own `output_schema.json` — the model is hard-constrained
to the schema regardless of what `other-grades/user.txt`'s final line says (see the note above).

In [ ]:
_VOCAB_COMPLEXITY_STEP_IDS = ["vocab_complexity_grades_3_4", "vocab_complexity_other_grades"]


def evaluate_complexity(text: str, grade: str, background_knowledge: str):
    condition_inputs = {"grade_level": grade}
    step = select_step(_VOCAB_COMPLEXITY_STEP_IDS, condition_inputs)
    llm = make_llm(step["spec"]["model"], step["spec"]["generation"]["temperature"])
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    inputs = {
        "text": text,
        "grade_level": grade,
        "student_background_knowledge": background_knowledge,
    }
    for prep in CONFIG.get("preprocessing", []):
        if prep["id"] == "fk_score" and condition_matches(prep.get("condition"), condition_inputs):
            inputs["fk_score"] = round(textstat.flesch_kincaid_grade(text), 2)

    prompt_template = ChatPromptTemplate.from_messages(step["messages"])
    rendered_messages = prompt_template.format_messages(**inputs)
    raw = structured_llm.invoke(rendered_messages)

    if raw.get("parsing_error"):
        raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

    return {
        "step_id": step["spec"]["id"],
        "rendered_prompt": [m.model_dump() for m in rendered_messages],
        "raw_output": raw["raw"],
        "raw_text": raw["raw"].content,
        "formatted_output": raw["parsed"],
        "usage": getattr(raw["raw"], "usage_metadata", None),
    }


### Put it together

In [ ]:
def evaluate_vocabulary(text: str, grade: str):
    """
    Evaluate vocabulary complexity for a text at a given grade, using the canonical
    config.json in this directory. grade_level determines which vocab_complexity step
    actually runs, via that step's own condition.

    Returns a dict with full I/O trace fields, same shape as the other example notebooks
    (rendered_prompt, raw_output, raw_text, formatted_output, usage), plus:
      - step_id:                 which vocab_complexity step actually ran
      - background_knowledge:    the step 1 output fed into step 2
    """
    try:
        background_knowledge = generate_background_knowledge(text, grade)
        result = evaluate_complexity(text, grade, background_knowledge)
        result["background_knowledge"] = background_knowledge
        return result
    except Exception as e:
        return f"Error evaluating text: {e}"


# Try evaluating text for vocabulary complexity

Evaluator may take up to a minute to run (two model calls: background knowledge, then complexity).

To evaluate your own text, replace the text and grade below and run the cell again.

In [ ]:
# Swap out text and grade here for your own text to evaluate

# Clear ID = 2204
sample_text = """
Polo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.
Some scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.
His father had borrowed money and bought a ship. He became wealthy because of his trading in the near East.
"""
grade = "3"

result = evaluate_vocabulary(sample_text, grade)

if isinstance(result, dict):
    out = result["formatted_output"]
    print(f"Step:                  {result['step_id']}")
    print(f"Background knowledge:  {result['background_knowledge']}")
    print(f"Complexity score:      {out['complexity_score']}")
    print(f"Tier 2 words:          {out['tier_2_words']}")
    print(f"Tier 3 words:          {out['tier_3_words']}")
    print(f"Archaic words:         {out['archaic_words']}")
    print(f"Other complex words:   {out['other_complex_words']}")
    print(f"\nReasoning:\n{out['reasoning']}")
else:
    print(result)


### Full I/O trace

In [ ]:
if isinstance(result, dict):
    print("=" * 60)
    print("RENDERED PROMPT (input sent to the vocab_complexity LLM)")
    print("=" * 60)
    pp.pprint(result["rendered_prompt"])

    print("\n" + "=" * 60)
    print("RAW LLM TEXT (model's verbatim output)")
    print("=" * 60)
    print(result["raw_text"])

    print("\n" + "=" * 60)
    print("PARSED OUTPUT (output_schema)")
    print("=" * 60)
    pp.pprint(result["formatted_output"])

    print("\n" + "=" * 60)
    print("USAGE METADATA")
    print("=" * 60)
    pp.pprint(result["usage"])
else:
    print(result)

### Sniff-test runner

Runs the fixtures from `fixtures.json` through `evaluate_vocabulary`, which routes each case to the
correct `vocab_complexity` step automatically via its `grade_level` input.


In [ ]:
fixtures_path = Path(CONFIG["fixtures"]["path"])
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

# Optional capture: set EVAL_CAPTURE_DIR to a directory to record the
# vocab_complexity step's rendered request + raw response for each fixture
# case, alongside its formatted output. NOTE: this captures only the second
# (vocab_complexity) step of the two-step pipeline -- the background_knowledge
# string is recorded as a plain input, not as its own full request/response
# trace, since generate_background_knowledge() doesn't return one today. Not
# wired into any check yet -- raw material for the planned replay-based
# contract tests. Unset by default, and always unset in CI.
_CAPTURE_DIR = os.environ.get("EVAL_CAPTURE_DIR")
captures = []

results = []
for fx in fixtures:
    out = evaluate_vocabulary(text=fx["input"]["text"], grade=fx["input"]["grade_level"])
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "detail": out})
        continue
    predicted = out["formatted_output"]
    mismatches = {
        field: (predicted[field], want)
        for field, want in fx["expected"].items()
        if predicted[field] != want
    }
    results.append({
        "id": fx["id"],
        "step_id": out["step_id"],
        "status": "pass" if not mismatches else "fail",
        "detail": mismatches or predicted["complexity_score"],
    })
    if _CAPTURE_DIR:
        step_spec = STEPS[out["step_id"]]["spec"]
        captures.append({
            "id": fx["id"],
            "step_id": out["step_id"],
            "background_knowledge": out["background_knowledge"],
            "rendered_prompt": out["rendered_prompt"],
            "raw_text": out["raw_text"],
            "formatted_output": out["formatted_output"],
            "model": step_spec["model"],
            "temperature": step_spec["generation"]["temperature"],
            "usage": out["usage"],
        })

print("=" * 88)
print(f"{'ID':>20}  {'STEP':<28}  {'STATUS':<8}  DETAIL")
print("=" * 88)
for r in results:
    print(f"{r['id']:>20}  {r.get('step_id','-'):<28}  {r['status'].upper():<8}  {r['detail']}")
print("=" * 88)
n_pass = sum(1 for r in results if r["status"] == "pass")
print(f"Summary: {n_pass} pass, {len(results) - n_pass} not passing  --  total {len(results)}")

if _CAPTURE_DIR:
    capture_path = Path(_CAPTURE_DIR) / "captures.json"
    capture_path.write_text(json.dumps(captures, indent=2) + "\n")
    print(f"\nWrote {len(captures)} captures to {capture_path.resolve()}")


You can copy or edit the above cells to test out different texts and grade levels.